In [7]:
import torch 
from torch import nn 
from torch.nn import functional as f 

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [9]:
device

'cuda'

In [10]:
# Dummy Model 
class Model(nn.Module): 
    def __init__(self): 
        super().__init__() 
        self.network = nn.Sequential(
            nn.Linear(2,4),
            nn.Linear(4,8) 
        )
    def forward(self, x): 
        return self.network(x)

In [11]:
model = Model().to(device)
c = nn.MSELoss()
optim = torch.optim.Adam(model.parameters() , lr=1e-3 ) 

##### When we train a transformer model , we consider couple of things :


1 - no one trains the model ops on fp32 🥀🥀 we train on fp16 or bf-16 Mixed precisions

In [12]:
x = torch.randn(1, 2).to(device)
y = torch.randn(1 , 8).to(device)
print(x)
for i in range(100): 
    optim.zero_grad() # reset gradients
    with torch.autocast(device_type="cuda" , dtype=torch.bfloat16):  # ops inside run in bf16 where safe; weights stay fp32
            y_pred = model(x)
            loss = c(y_pred,y).to(device)

    if i % 20 == 0: 
        print(f"Loss : {loss}" ) 
    loss.backward()
    optim.step()
    

tensor([[-0.7346,  0.3612]], device='cuda:0')
Loss : 2.10611629486084
Loss : 1.9300713539123535
Loss : 1.7763545513153076
Loss : 1.6364450454711914
Loss : 1.5061334371566772


2 - Using Warmups + Decay

In [19]:
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

In [23]:
optim = torch.optim.AdamW(model.parameters(), lr=3e-3)

warmup = LinearLR( 
    start_factor=0.01,
    end_factor=1,
    optimizer=optim,
    total_iters=10
)

decay = CosineAnnealingLR(
    T_max=100,
    optimizer=optim,
    eta_min=1e-6, 
    
)

schedular = SequentialLR(optimizer=optim , milestones=[10],schedulers=[warmup,decay]   
)


When we train , we do : 👌🙌

In [29]:
x = torch.randn(1, 2).to(device)
y = torch.randn(1 , 8).to(device)
print(x)
for i in range(100): 
    optim.zero_grad() # reset gradients
    with torch.autocast(device_type="cuda" , dtype=torch.bfloat16):  # ops inside run in bf16 where safe; weights stay fp32
            y_pred = model(x)
            loss = c(y_pred,y).to(device)

    if i % 20 == 0: 
        print(f"Loss : {loss}" ) 
    loss.backward()
    optim.step()
    schedular.step() # we add our decay and warmup
    
    
    

tensor([[-0.9314,  1.0514]], device='cuda:0')
Loss : 2.0385665893554688
Loss : 1.2813024520874023
Loss : 0.8568096160888672
Loss : 0.6605962514877319
Loss : 0.5694262981414795


3 - Custom AdamW: because why the hell should we apply weight decay to parameters that shouldn't be regularized?

In [63]:
decay_params = [] 
non_decay_params = [] 

params = {idx: p for idx , p in model.named_parameters()}

for idx , p in params.items(): 
    if not p.requires_grad : 
        continue

    if p.dim() == 1 or idx.endswith(".bias"): 
        non_decay_params.append(p) 
    else: 
        decay_params.append(p) 


In [64]:
decay_params

[Parameter containing:
 tensor([[-0.0828, -0.7656],
         [ 0.0352, -0.1617],
         [-0.4944, -0.1566],
         [ 0.3000,  0.4759]], device='cuda:0', requires_grad=True),
 Parameter containing:
 tensor([[-0.1444, -0.4582, -0.2366,  0.1726],
         [-0.0685,  0.0677, -0.0611, -0.2487],
         [ 0.0481, -0.2076, -0.1050,  0.0246],
         [ 0.3836, -0.2020,  0.7107,  0.2792],
         [ 0.3587, -0.2397,  0.1609,  0.2225],
         [ 0.1437, -0.3843, -0.1250,  0.3859],
         [ 0.6066, -0.0248,  0.0388,  0.1475],
         [ 0.3493, -0.3752, -0.2376, -0.0452]], device='cuda:0',
        requires_grad=True)]

In [65]:
non_decay_params

[Parameter containing:
 tensor([ 0.0076, -0.5744, -0.0566,  0.3811], device='cuda:0',
        requires_grad=True),
 Parameter containing:
 tensor([ 0.1677,  0.1285, -0.5942, -0.2514, -0.1163,  0.3000, -0.7131, -0.0047],
        device='cuda:0', requires_grad=True)]

In [67]:
optim = torch.optim.AdamW(
        [
            {"params" : decay_params , "weight_decay" : 0.1},
            {"params" : non_decay_params , "weight_decay" : 0.0},
        ]
    
    , lr=3e-3)

warmup = LinearLR( 
    start_factor=0.01,
    end_factor=1,
    optimizer=optim,
    total_iters=10
)

decay = CosineAnnealingLR(
    T_max=100,
    optimizer=optim,
    eta_min=1e-6, 
    
)

schedular = SequentialLR(optimizer=optim , milestones=[10],schedulers=[warmup,decay]   
)


In [70]:
x = torch.randn(1, 2).to(device)
y = torch.randn(1 , 8).to(device)
print(x)
for i in range(100): 
    optim.zero_grad() # reset gradients
    with torch.autocast(device_type="cuda" , dtype=torch.bfloat16):  # ops inside run in bf16 where safe; weights stay fp32
            y_pred = model(x)
            loss = c(y_pred,y).to(device)

    if i % 20 == 0: 
        print(f"Loss : {loss}" ) 
    loss.backward()
    optim.step()
    schedular.step() # we add our decay and warmup
    

tensor([[-2.5121, -0.0196]], device='cuda:0')
Loss : 0.6037207841873169
Loss : 0.17643795907497406
Loss : 0.02507534995675087
Loss : 0.005230960436165333
Loss : 0.0023314659483730793
